# CAPTCHA Farm Attack Simulation (RQ2: token reuse / relay)

Evaluates the context-bound, single-use tokens of the **revised** protocol in `thermocaptcha_protocol.py`. `Final_code.ipynb` uses the same module.

**Scenario.** A farm *worker* (persistent device key, IP, User-Agent) legitimately solves N challenges and obtains tokens $T_W$. For each token, a farm *customer* (attacker) redeems it at the real website from their own environment. The website observes the attacker's connection and forwards it with a valid HMAC proof. The server recomputes $F_{\rm current}=H(IP_{\rm cur}\parallel UA_{\rm cur}\parallel H(pk_{\rm cur}))$ and requires $F_{\rm current}=F_{\rm token}=F_{\rm session}$.

Attacker strategies are rotated over the attempts (25% each):
| Strategy | IP | User-Agent | Key used for redemption proof |
|---|---|---|---|
| S1 full relay | attacker | attacker | attacker key |
| S2 spoofed UA | attacker | **worker** | attacker key |
| S3 proxied IP + spoofed UA | **worker** | **worker** | attacker key |
| S4 copied public key | **worker** | **worker** | presents **worker pk**, cannot sign with worker sk |

**Controls** (not counted as attacks): after each relay attempt, the worker redeems its own token from its own context (must be **accepted**, which shows that failed relays do not burn the token). The worker then replays it (must be **rejected**, single use).

N ∈ {100, 500, 1000} → 1,600 reuse attempts. Each token is relayed immediately after issuance, so no rejection is caused by token expiry.

## Setup

In [1]:
import json, os, shutil, time, base64, random, collections, uuid
import pandas as pd
import thermocaptcha_protocol as tp

SOURCE_DB = "thermal_captcha.db"
EVAL_DB = "rq2_farm_eval.db"          # fresh copy per run -> reproducible, does not touch the shipped DB
TEST_IMAGE_PATH = "./25.bmp"
SITE_KEY = "28024d2408754883a7eff1db9cea2b00"
WEBSITE_NAME = "microsoft.com"
random.seed(2026)

shutil.copy(SOURCE_DB, EVAL_DB)
tp.ensure_demo_secrets(SITE_KEY)
server = tp.ThermoCaptchaServer(EVAL_DB)                      # real thermal detector (thermalImageChecker)
website = tp.Website(SITE_KEY, WEBSITE_NAME, tp.load_website_secret())
with open(TEST_IMAGE_PATH, "rb") as f:
    IMAGE = f.read()

def new_device():
    # Simulated devices use in-memory persistent keys (one key per device, reused across challenges).
    return tp.ThermoCaptchaClient(tp.rsa.generate_private_key(public_exponent=65537, key_size=2048))

print("Protocol:", tp.PROTOCOL_VERSION, "| eval DB:", EVAL_DB)

TEST_VALUES = [100, 500, 1000]
WORKER_IP = "192.168.1.100"
WORKER_UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
ATTACKER_IP_BASE = "10.0.0."
ATTACKER_UAS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 14_0 like Mac OS X) AppleWebKit/605.1.15",
]
STRATEGIES = ["S1 full relay", "S2 spoofed UA", "S3 proxied IP + spoofed UA", "S4 copied public key"]

worker = new_device()
attacker_pool = [new_device() for _ in range(20)]     # farm customers' own devices

NOTE: generated ephemeral demo secrets for: THERMOCAPTCHA_SERVER_SECRET, THERMOCAPTCHA_WEBSITE_SECRET, THERMOCAPTCHA_SITE_SECRETS[28024d2408754883a7eff1db9cea2b00] (set them in the environment for a real deployment).
Protocol: thermocaptcha-v2 | eval DB: rq2_farm_eval.db


## Worker and attacker

In [2]:
def worker_solve_challenge():
    ch = server.issue_challenge(SITE_KEY)
    res = server.verify_capture(worker.build_capture_payload(ch, IMAGE), WORKER_IP, WORKER_UA)
    return res

def attacker_reuse_token(token_response_json, i):
    strategy = STRATEGIES[i % 4]
    attacker = random.choice(attacker_pool)
    ip, ua = f"{ATTACKER_IP_BASE}{random.randint(1, 254)}", random.choice(ATTACKER_UAS)
    user_payload = json.loads(attacker.build_redemption_payload(token_response_json))
    if strategy == "S2 spoofed UA":
        ua = WORKER_UA
    elif strategy == "S3 proxied IP + spoofed UA":
        ip, ua = WORKER_IP, WORKER_UA
    elif strategy == "S4 copied public key":
        ip, ua = WORKER_IP, WORKER_UA
        user_payload["public_key"] = worker.public_key_pem   # public, so copyable; private key is not
    req = website.build_server_request(json.dumps(user_payload), ip, ua)
    return strategy, server.redeem_token(req)

def worker_redeem(token_response_json):
    return server.redeem_token(website.build_server_request(
        worker.build_redemption_payload(token_response_json), WORKER_IP, WORKER_UA))

## Simulation

In [3]:
def simulate_captcha_farm_attack(n_tokens):
    tokens = accepted = 0
    reasons = collections.Counter(); by_strategy = collections.Counter()
    legit_ok = replay_ok = 0
    t0 = time.time()
    for i in range(n_tokens):
        res = worker_solve_challenge()
        if not res.ok:
            raise RuntimeError(f"worker could not obtain a token: {res.reason}")
        tokens += 1
        strategy, attack = attacker_reuse_token(res.data["response_json"], i)
        accepted += int(attack.ok)
        reasons[attack.reason] += 1
        by_strategy[(strategy, attack.ok)] += 1
        legit_ok += int(worker_redeem(res.data["response_json"]).ok)      # control: must be accepted
        replay_ok += int(worker_redeem(res.data["response_json"]).ok)     # control: must be rejected
    print(f"N={n_tokens}: tokens={tokens} reuse attempts={tokens} accepted={accepted} | "
          f"worker redemption accepted={legit_ok}/{tokens} | worker replay accepted={replay_ok} "
          f"({time.time()-t0:.1f}s)")
    return {"N": n_tokens, "Tokens Collected": tokens, "Reuse Attempts": tokens, "Accepted": accepted,
            "Success Rate (%)": 100.0 * accepted / tokens,
            "Legit redemption accepted": legit_ok, "Worker replay accepted": replay_ok,
            "reasons": dict(reasons),
            "by_strategy": {s: by_strategy[(s, False)] + by_strategy[(s, True)] for s in STRATEGIES},
            "accepted_by_strategy": {s: by_strategy[(s, True)] for s in STRATEGIES}}

results = [simulate_captcha_farm_attack(n) for n in TEST_VALUES]

N=100: tokens=100 reuse attempts=100 accepted=0 | worker redemption accepted=100/100 | worker replay accepted=0 (1.0s)


N=500: tokens=500 reuse attempts=500 accepted=0 | worker redemption accepted=500/500 | worker replay accepted=0 (4.5s)


N=1000: tokens=1000 reuse attempts=1000 accepted=0 | worker redemption accepted=1000/1000 | worker replay accepted=0 (8.9s)


## Results summary (1,600 token-reuse attempts)

In [4]:
df_results = pd.DataFrame(results)
print(df_results[["N", "Tokens Collected", "Reuse Attempts", "Accepted", "Success Rate (%)",
                  "Legit redemption accepted", "Worker replay accepted"]].to_string(index=False))
total_attempts = int(df_results["Reuse Attempts"].sum()); total_accepted = int(df_results["Accepted"].sum())
print(f"\nTotal Reuse Attempts: {total_attempts}\nTotal Accepted: {total_accepted}\n"
      f"Overall Success Rate: {100.0*total_accepted/total_attempts:.2f}%")

print("\nAttempts / accepted by strategy (all N):")
for s in STRATEGIES:
    n = sum(r["by_strategy"][s] for r in results); a = sum(r["accepted_by_strategy"][s] for r in results)
    print(f"  {s:<28} attempts={n:<5} accepted={a}")
print("\nRejection reasons (all N):")
agg = collections.Counter()
for r in results:
    agg.update(r["reasons"])
for k, v in agg.items():
    print(f"  {v:>5}  {k}")
server.close()

   N  Tokens Collected  Reuse Attempts  Accepted  Success Rate (%)  Legit redemption accepted  Worker replay accepted
 100               100             100         0               0.0                        100                       0
 500               500             500         0               0.0                        500                       0
1000              1000            1000         0               0.0                       1000                       0

Total Reuse Attempts: 1600
Total Accepted: 0
Overall Success Rate: 0.00%

Attempts / accepted by strategy (all N):
  S1 full relay                attempts=400   accepted=0
  S2 spoofed UA                attempts=400   accepted=0
  S3 proxied IP + spoofed UA   attempts=400   accepted=0
  S4 copied public key         attempts=400   accepted=0

Rejection reasons (all N):
   1200  context mismatch: F_current != F_token or F_token != F_session
    400  invalid redemption proof (key possession)
